# CDS524 Assignment 1 — AI Snake (DQN) (Colab Notebook)

> **Goal**: Provide a reproducible Google Colab workflow **with in-line explanations** for the Snake + DQN implementation in `snake_dqn.py`.

> **Note**: Colab is not ideal for rendering an interactive Pygame window. This notebook runs **headless training** (`HEADLESS=1`) and verifies / visualizes the generated artifacts.

## What you will run
- **Training**: `HEADLESS=1 python snake_dqn.py` (runs training and exits)
- **Artifacts**: `model.pth` (checkpoint) and `training_curve.png` (learning curve)

## Method summary (DQN)
### State $s$ (11-D binary features)
The agent observes an 11-dimensional state vector:

| Index | Feature | Meaning |
|---:|---|---|
| 0 | `danger_straight` | Collision if keep moving straight |
| 1 | `danger_right` | Collision if turn right |
| 2 | `danger_left` | Collision if turn left |
| 3 | `food_left` | Food is left of head |
| 4 | `food_right` | Food is right of head |
| 5 | `food_up` | Food is above head |
| 6 | `food_down` | Food is below head |
| 7 | `dir_left` | Current moving direction is left |
| 8 | `dir_right` | Current moving direction is right |
| 9 | `dir_up` | Current moving direction is up |
| 10 | `dir_down` | Current moving direction is down |

### Action $a$ (relative action space, 3 actions)
Actions are relative to the current heading:
- `0`: **Left** turn
- `1`: **Straight**
- `2`: **Right** turn

### Reward $r$ (dense + terminal)
The environment uses a shaped reward:
- Small living reward (`+0.1`) per step
- Distance shaping: if the head gets farther from food, apply a penalty (`-0.5`)
- Collision / death: `-10` and episode ends
- Eating food: reward depends on food type (normal/bonus/poison) and includes combo bonus

### Q-network
A simple MLP: `11 → 256 → 128 → 64 → 3` with ReLU, trained with:
- Replay buffer (maxlen 10k), batch size 32
- MSE loss on TD target
- Adam (lr=1e-3), discount factor $\gamma=0.95$
- Epsilon-greedy exploration with decay (`0.8 → 0.01`)


## Step 1 — Install dependencies
This installs the minimal dependencies to run `snake_dqn.py` in Colab. If you cloned the repo (Step 2 option B) and have `requirements.txt`, you can switch to installing from that file.

In [ ]:
# Install dependencies (quietly).
# If you have requirements.txt in the current folder, you can install from it instead.
import os

if os.path.exists('requirements.txt'):
    !pip -q install -r requirements.txt
else:
    !pip -q install pygame torch matplotlib

import sys, torch, pygame
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('Pygame:', pygame.version.ver)

## Step 2 — Get the code
You have two equivalent options. Pick **one**:

**Option A (recommended for LMS submission): upload the files**
- Upload at least: `snake_dqn.py`
- Optional: `requirements.txt` (then Step 1 installs from it)

**Option B: clone from GitHub**
- Useful if the repo is public/accessible to the grader
- The cell below auto-finds `snake_dqn.py` after cloning

In [ ]:
# Choose ONE of the following options.
USE_GITHUB = False  # set True to clone from GitHub instead of uploading
GITHUB_REPO = 'https://github.com/rainy-sy/CDS524-Assignment1-ai-snake-dqn.git'

import os, glob, shutil

if USE_GITHUB:
    # Clone repo fresh
    repo_dir = '/content/CDS524-Assignment1-ai-snake-dqn'
    if os.path.exists(repo_dir):
        shutil.rmtree(repo_dir)
    !git clone -q {GITHUB_REPO} {repo_dir}
    os.chdir(repo_dir)
else:
    # Upload files from your computer (snake_dqn.py is required)
    from google.colab import files
    files.upload()

# Auto-locate snake_dqn.py (supports either upload or repo clone)
candidates = [p for p in glob.glob('**/snake_dqn.py', recursive=True) if not p.startswith('.venv')]
print('Found candidates:', candidates[:10])
assert len(candidates) > 0, 'snake_dqn.py not found. Upload it or enable USE_GITHUB.'

# Prefer the assignment version if present
preferred = None
for p in candidates:
    if 'Assignment 1' in p and 'WANG Shiyu' in p:
        preferred = p
        break
SNAKE_SCRIPT = preferred or candidates[0]
print('Using script:', SNAKE_SCRIPT)

## Step 3 — Run headless training
This runs the training loop in **headless mode** so it works in Colab. The script will:
- Run ~300 episodes (default)
- Save `model.pth` and `training_curve.png` into the current working directory
- Exit automatically when done (in headless mode)

If you want to play interactively with graphics, run locally on your machine (see `README.md`).

In [ ]:
# Run training headlessly (no GUI).
# The environment variable HEADLESS=1 makes snake_dqn.py use SDL 'dummy' drivers.
!HEADLESS=1 python {SNAKE_SCRIPT}

## Step 4 — Inspect artifacts
After training, you should have:
- `training_curve.png` (plot of score over episodes + moving average)
- `model.pth` (PyTorch checkpoint)

The following cells verify files exist, visualize the curve, and optionally download artifacts.

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

print('training_curve.png exists?', os.path.exists('training_curve.png'))
print('model.pth exists?', os.path.exists('model.pth'))

if os.path.exists('training_curve.png'):
    img = mpimg.imread('training_curve.png')
    plt.figure(figsize=(10, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Curve')
    plt.show()

In [ ]:
# Optional: download artifacts to your computer
# (Uncomment what you need.)
from google.colab import files

# if os.path.exists('model.pth'):
#     files.download('model.pth')
# if os.path.exists('training_curve.png'):
#     files.download('training_curve.png')


---
## Local interactive demo (optional)
If you need an interactive demo video, it is best to record locally (not Colab):
- Install deps: `pip install -r requirements.txt`
- Run: `python snake_dqn.py`
- Controls: `F1` manual, `F2` random_ai, `F3` training, `F4` test; `T` cycles (manual→random_ai→test); `SPACE` pause; `R` reset